# SARA medium (~132M) — Kaggle training on YOUR full Drive (4 GB)
**Kaggle:** Settings → Accelerator → **GPU T4 x2** (ya P100). Internet: **ON**.

**Plan:** 4 GB text ≈ ~1B tokens. Chinchilla-optimal for 132M params = ~2.6B tokens = multiple epochs over your data.

| Cell | Kaam | Time |
|---|---|---|
| 1 | Clone + install | 3 min |
| 2 | PDFs (dataset ya Drive link) | 1 min |
| 3 | Corpus build | ~15 min (bada corpus) |
| 4 | **Training 40k steps** | **~12–16 hr** — 2 din mein 2 sessions, checkpoint resume se continue |
| 5 | Generate | 10s |
| 6 | Output save | 10s |

**Session khatam hone pe:** Cell 7 (resume) use karo naya session shuru karne ke liye.

In [ ]:
# 1. Clone + deps
!git clone https://github.com/skmandal3240/SARA /kaggle/working/SARA
%cd /kaggle/working/SARA
!pip install -q -r requirements.txt pyyaml pymupdf gdown

In [ ]:
# 2. PDFs — dataset zip YA public Drive folder
import glob, os, subprocess
os.makedirs('/kaggle/working/pdfs', exist_ok=True)
zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
if zips:
    subprocess.run(['unzip','-o','-q',zips[0],'-d','/kaggle/working/pdfs'])
else:
    FOLDER_URL = 'https://drive.google.com/drive/folders/1OQk3FKLxI1GSVjf5ViW7nKu_x-oC3854'  # <- apna bada folder ka link yahan daalo
    r = subprocess.run(['gdown','--folder',FOLDER_URL,'-O','/kaggle/working/pdfs'], capture_output=True, text=True)
    print(r.stdout[-300:], r.stderr[-300:])
pdfs = glob.glob('/kaggle/working/pdfs/**/*.pdf', recursive=True)
print(f'{len(pdfs)} PDFs')
assert pdfs, 'No PDFs found'

In [ ]:
# 3. Corpus build (PDF -> text -> train.bin). Bade corpus pe thoda time lagega.
!python scripts/pdf_to_corpus.py /kaggle/working/pdfs
!python prepare_data.py --config configs/sara_medium.yaml --vocab-size 16384 --source file:data/drive_corpus.txt
!python -c "import os; print('corpus MB:', os.path.getsize('data/drive_corpus.txt')//1048576); print('train.bin MB:', os.path.getsize('data/train.bin')//1048576)"

In [ ]:
# 4. TRAIN medium — 40k steps. Checkpoint har 2000 steps pe save hota hai.
#    Ek session ~9h = ~25-30k steps. Baaki ke liye session dobara start karke Cell 7 chalao.
!python train_medium.py --steps 40000 --batch 8 --seq 512

In [ ]:
# 5. Generate from trained model
import torch
from pathlib import Path
from generate import load_sara

model, tok, cfg = load_sara(Path('checkpoints/sara_medium/sara.pt'))
for p in ['The sun is', 'Computer science is', 'Machine learning models']:
    ids = torch.tensor([tok.encode(tok.wrap_user(p))], dtype=torch.long)
    out = model.generate(ids, max_new=80, temperature=0.8, eos_id=tok.eos_id)
    print(f'>>> {p}\n{tok.decode(out[0].tolist())}\n')

In [ ]:
# 6. Save output
import shutil
shutil.copy('checkpoints/sara_medium/sara.pt', '/kaggle/working/sara_medium_trained.pt')
shutil.copy('tokenizer/sara.json', '/kaggle/working/sara_tokenizer_16k.json')
print('Output tab -> download sara_medium_trained.pt (~520 MB)')

## 7. Resume after session restart

In [ ]:
# RESUME: naya session start hone pe cells 1-3 chala ke phir ye chalao.
# Purana checkpoint Kaggle Dataset mein re-upload karo ('sara-medium-ckpt'), phir:
import shutil, glob
ck = glob.glob('/kaggle/input/sara-medium-ckpt/*.pt')
if ck:
    os.makedirs('checkpoints/sara_medium', exist_ok=True)
    shutil.copy(ck[0], 'checkpoints/sara_medium/sara.pt')
    # NOTE: train_medium.py abhi fresh se start karta hai; optimizer state bhi save
    # karne wala resume support agle version mein. Filhal har session ko poora run
    # mat do — steps kam rakhke zyada sessions use karo.
!python train_medium.py --steps 40000 --batch 8